In [2]:
import os
import ee
import geemap
from dotenv import load_dotenv

from tools.Input_values.satellites import sentinel2_harmonized
from tools.Input_values.locaties import select_naturenreserves, build_location_bound, RegionMode
from tools.data_cleansing.sentinel2 import add_local_cloud_cover, scl_cloud_removal_20m, scale_reflectance
from tools.calculate_spectral_indices.sentinel2 import calc_ndvi, calc_evi

In [3]:
load_dotenv()

project_code_ee = os.getenv("PROJECT_CODE_EE")

if not project_code_ee:
    raise RuntimeError("PROJECT_CODE_EE not found")

ee.Authenticate()
ee.Initialize(project=project_code_ee)

In [4]:
natuurgebieden = select_naturenreserves(["Nieuwkoopse Plassen & de Haeck"])
type_coverage : RegionMode = "bounds"
buffer_size_meters : int = 0

filter_date_start : str = "2020-07-10"
filter_date_end : str = "2020-07-12"
local_cloud_cover_threshold : int = 30

In [5]:
s2 = (
    ee.ImageCollection(sentinel2_harmonized)
    .filterDate("2020-07-10", "2020-07-12")
    .filterBounds(natuurgebieden)
    .map(add_local_cloud_cover(natuurgebieden, coverage = "bounds"))
    .map(scl_cloud_removal_20m(remove_unclassified=False, bloat_n_meters= 50))
    .map(scale_reflectance)
    .map(calc_ndvi)
    .map(calc_evi)
)

In [6]:
from tools.extract_downloads.geemap_image import download_single_geotiff

download_single_geotiff(
    image=s2.first(),
    filename="output/ndvi_first_image.tif",
    locations_of_interest=natuurgebieden,
    coverage="buffer",
    buffer_meters= 100,
    bands=["NDVI"],
    scale=10,
)

Generating URL ...
Please wait ...
Data downloaded to c:\AAA Python\Remote Sensing BO\output\ndvi_first_image.tif


In [7]:
from tools.extract_downloads.geemap_image import convert_geotiff_to_visual_image

visNDVI = {
    'bands': ['NDVI'],
    'min': -1,
    'max': 1,
    'palette': ['red', 'yellow', 'green']
}

convert_geotiff_to_visual_image(
    tif_filename=r"output\ndvi_first_image.tif",
    output_filename=r"output\ndvi_first_image.png",
    vis_params= visNDVI

)

In [8]:
import pandas as pd
locations_df = pd.read_csv(r"Inputdata\PQ_coordinaat_datum_Sjoerd.csv")

pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


def join_unique_dates(series):
    dates = pd.to_datetime(series.astype(str), format="%Y%m%d", errors="coerce")

    dates = dates.dropna().drop_duplicates().sort_values()

    return ", ".join(dates.dt.strftime("%Y-%m-%d"))

unique_groups = (
    locations_df
    .groupby(["PQ_nummer", "X", "Y"], as_index=False)
    .agg({
        "Datum": join_unique_dates
    })
)

len(unique_groups)
unique_groups

,PQ_nummer,X,Y,Datum
0,1112459132,112222,459541,"2020-08-10, 2023-08-09"
1,1112459132,112226,459552,"2002-08-29, 2006-09-13, 2010-09-21, 2014-09-25, 2017-09-22"
2,1113459052,113302,459476,"2002-08-29, 2006-09-13, 2010-09-21, 2014-09-25, 2017-09-22, 2020-08-10, 2023-08-09"
3,1113460072,113567,460859,"2002-08-29, 2006-09-13, 2010-09-21, 2014-09-25, 2017-09-22"
4,1113460072,113579,460868,"2020-08-10, 2023-08-09"
5,1113460777,113030,460838,"2007-04-23, 2011-05-02, 2016-05-04, 2018-06-04, 2021-06-08, 2024-06-07"
6,1113461012,113682,461789,"2002-08-29, 2005-08-24, 2009-08-31, 2013-08-29, 2016-09-19, 2019-08-30, 2022-09-22, 2025-09-11"
7,1113461022,113694,461773,"2009-08-31, 2013-08-30, 2016-09-19, 2019-08-30, 2022-09-22, 2025-09-11"
8,1113461032,113697,461759,"2009-08-31, 2013-08-30, 2016-09-19, 2019-08-30, 2022-09-22, 2025-09-11"
9,1113461042,113745,461716,"2019-08-30, 2022-09-22, 2025-09-11"


In [9]:
import math
cell_size = 100  # meters
rotation_deg = 30  # clockwise rotation

reference_type = "center"   # "center" or "corner"
corner_type = "SW"          # "SW", "SE", "NW", "NE" (only used if reference_type="corner"



In [10]:
import pandas as pd
import numpy as np
from typing import Literal

PointType = Literal["center", "top_left", "top_right", "bottom_right", "bottom_left"]
VALID_POINT_TYPES = ("center", "top_left", "top_right", "bottom_right", "bottom_left")

def rotate_point(x, y, cx, cy, angle_rad):
    cos_a = math.cos(angle_rad)
    sin_a = math.sin(angle_rad)

    dx = x - cx
    dy = y - cy

    rx = cx + dx * cos_a - dy * sin_a
    ry = cy + dx * sin_a + dy * cos_a

    return rx, ry

def generate_rectangle_corners(
    x,
    y,
    width,
    height,
    rotation_deg=0,
    point_type: PointType = "center"
):
    if point_type not in VALID_POINT_TYPES:
        raise ValueError(f"point_type moet één van deze waarden zijn: {VALID_POINT_TYPES}")

    if width <= 0 or height <= 0:
        raise ValueError("width en height moeten groter zijn dan 0")

    angle = np.radians(rotation_deg)
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)

    local = {
        "top_left": (-width / 2, height / 2),
        "top_right": (width / 2, height / 2),
        "bottom_right": (width / 2, -height / 2),
        "bottom_left": (-width / 2, -height / 2),
    }

    def rotate(dx, dy):
        rx = dx * cos_a - dy * sin_a
        ry = dx * sin_a + dy * cos_a
        return rx, ry

    if point_type == "center":
        cx, cy = x, y
    else:
        dx, dy = local[point_type]
        rdx, rdy = rotate(dx, dy)
        cx = x - rdx
        cy = y - rdy

    corners = {}
    for name, (dx, dy) in local.items():
        rdx, rdy = rotate(dx, dy)
        corners[name] = (cx + rdx, cy + rdy)

    return corners


def corners_to_columns(
    row,
    x_col="X",
    y_col="Y",
    width=10,
    height=10,
    rotation_deg=0,
    point_type="center"
):
    if x_col not in row.index:
        raise KeyError(f"Kolom '{x_col}' bestaat niet in de DataFrame")
    if y_col not in row.index:
        raise KeyError(f"Kolom '{y_col}' bestaat niet in de DataFrame")

    corners = generate_rectangle_corners(
        x=row[x_col],
        y=row[y_col],
        width=width,
        height=height,
        rotation_deg=rotation_deg,
        point_type=point_type,
    )

    return pd.Series({
        "x_tl": corners["top_left"][0],
        "y_tl": corners["top_left"][1],
        "x_tr": corners["top_right"][0],
        "y_tr": corners["top_right"][1],
        "x_br": corners["bottom_right"][0],
        "y_br": corners["bottom_right"][1],
        "x_bl": corners["bottom_left"][0],
        "y_bl": corners["bottom_left"][1],
    })

In [11]:
import pandas as pd
locations_df = pd.read_csv(r"Inputdata\PQ_coordinaat_datum_Sjoerd.csv")

pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


def join_unique_dates(series):
    dates = pd.to_datetime(series.astype(str), format="%Y%m%d", errors="coerce")

    dates = dates.dropna().drop_duplicates().sort_values()

    return ", ".join(dates.dt.strftime("%Y-%m-%d"))

unique_groups = (
    locations_df
    .groupby(["PQ_nummer", "X", "Y"], as_index=False)
    .agg({
        "Datum": join_unique_dates
    })
)


# Bereken de hoekcoördinaten voor elke rij
corner_cols = unique_groups.apply(
    corners_to_columns,
    x_col="X",
    y_col="Y",
    axis=1,
    width=10,
    height=10,
    rotation_deg=0,
    point_type="center"
)

# Voeg de nieuwe hoekkolommen toe aan de originele DataFrame
df = pd.concat([unique_groups, corner_cols], axis=1)



In [12]:
import ee
import pandas as pd

ee.Initialize()


def extract_imagecollection_stats(
    df: pd.DataFrame,
    image_collection: ee.ImageCollection,
    band_names,
    description="ee_plot_stats_export",
    scale=10,
    crs="EPSG:28992",
    export="drive",
    folder=None,
    file_name_prefix=None,
    include_mean=True,
    include_median=True,
    include_percentiles=(25, 75),
    include_stddev=True,
    include_minmax=False,
    include_count=True,
    keep_geometry=False,
):
    """
    Zet een pandas DataFrame met vierkanten om naar een Earth Engine FeatureCollection,
    reduceert alle images in een ImageCollection over deze polygonen,
    en start een export van de resultaten als CSV.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame met minimaal:
        x_tl, y_tl, x_tr, y_tr, x_br, y_br, x_bl, y_bl
    image_collection : ee.ImageCollection
        De collectie die je wilt analyseren.
    band_names : list[str] | tuple[str] | str
        Band of banden om te reduceren.
    description : str
        Naam van de Earth Engine exporttaak.
    scale : int | float
        Pixel scale voor reduction.
    crs : str
        CRS van zowel de input polygonen als reduction, bijv. 'EPSG:28992'.
    export : str
        'drive' of 'asset' (hier alleen drive uitgewerkt).
    folder : str | None
        Google Drive folder.
    file_name_prefix : str | None
        Prefix van exportbestand.
    include_mean : bool
    include_median : bool
    include_percentiles : tuple/list
        Bijvoorbeeld (25, 75) of (25, 50, 75)
    include_stddev : bool
    include_minmax : bool
    include_count : bool
    keep_geometry : bool
        Of geometrie in output moet blijven.

    Returns
    -------
    dict met:
        - feature_collection
        - result_feature_collection
        - task
    """

    if isinstance(band_names, str):
        band_names = [band_names]
    band_names = list(band_names)

    required_cols = [
        "x_tl", "y_tl",
        "x_tr", "y_tr",
        "x_br", "y_br",
        "x_bl", "y_bl"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Ontbrekende kolommen in df: {missing}")

    # Alle niet-hoekkolommen als properties meenemen
    property_cols = [c for c in df.columns if c not in required_cols]

    def row_to_feature(row):
        coords = [[
            [float(row["x_tl"]), float(row["y_tl"])],
            [float(row["x_tr"]), float(row["y_tr"])],
            [float(row["x_br"]), float(row["y_br"])],
            [float(row["x_bl"]), float(row["y_bl"])],
            [float(row["x_tl"]), float(row["y_tl"])],  # sluit polygon
        ]]

        geom = ee.Geometry.Polygon(coords, proj=crs, geodesic=False)

        props = {"row_id": int(row.name)}
        for col in property_cols:
            value = row[col]

            # pandas/numpy types veilig omzetten
            if pd.isna(value):
                props[col] = None
            elif isinstance(value, (int, float, str, bool)):
                props[col] = value
            else:
                props[col] = str(value)

        return ee.Feature(geom, props)

    features = [row_to_feature(row) for _, row in df.iterrows()]
    plots_fc = ee.FeatureCollection(features)

    # Reducer opbouwen
    reducer = None

    def add_reducer(base, new):
        if base is None:
            return new
        return base.combine(new, sharedInputs=True)

    if include_mean:
        reducer = add_reducer(reducer, ee.Reducer.mean())

    if include_median:
        reducer = add_reducer(reducer, ee.Reducer.median())

    if include_percentiles and len(include_percentiles) > 0:
        reducer = add_reducer(
            reducer,
            ee.Reducer.percentile(list(include_percentiles))
        )

    if include_stddev:
        reducer = add_reducer(reducer, ee.Reducer.stdDev())

    if include_minmax:
        reducer = add_reducer(reducer, ee.Reducer.minMax())

    if include_count:
        reducer = add_reducer(reducer, ee.Reducer.count())

    if reducer is None:
        raise ValueError("Er is geen reducer geselecteerd.")

    ic = image_collection.select(band_names)

    def reduce_image(img):
        date = ee.Date(img.get("system:time_start")).format("YYYY-MM-dd")

        reduced = img.reduceRegions(
            collection=plots_fc,
            reducer=reducer,
            scale=scale,
            crs=crs
        )

        def set_meta(f):
            out = f.set({
                "image_id": img.id(),
                "date": date,
                "system_time_start": img.get("system:time_start")
            })
            return out if keep_geometry else out.setGeometry(None)

        return reduced.map(set_meta)

    result_fc = ic.map(reduce_image).flatten()

    if export.lower() != "drive":
        raise NotImplementedError("Op dit moment is alleen export='drive' uitgewerkt.")

    export_args = {
        "collection": result_fc,
        "description": description,
        "fileFormat": "CSV",
    }

    if folder is not None:
        export_args["folder"] = folder

    if file_name_prefix is not None:
        export_args["fileNamePrefix"] = file_name_prefix

    task = ee.batch.Export.table.toDrive(**export_args)
    task.start()

    return {
        "feature_collection": plots_fc,
        "result_feature_collection": result_fc,
        "task": task
    }

Gebruik:

sampleRegions() voor punten
reduceRegions() voor polygonen/vakken

In [13]:
# 1. Eerst FeatureCollection maken is niet nodig;
#    de functie doet dat zelf uit df.

# 2. Tijdelijke bounds maken uit df kan handig zijn voor filterBounds:
def df_to_fc(df, crs="EPSG:28992"):
    feats = []
    for idx, row in df.iterrows():
        coords = [[
            [float(row["x_tl"]), float(row["y_tl"])],
            [float(row["x_tr"]), float(row["y_tr"])],
            [float(row["x_br"]), float(row["y_br"])],
            [float(row["x_bl"]), float(row["y_bl"])],
            [float(row["x_tl"]), float(row["y_tl"])],
        ]]
        geom = ee.Geometry.Polygon(coords, proj=crs, geodesic=False)
        feats.append(ee.Feature(geom, {"row_id": int(idx)}))
    return ee.FeatureCollection(feats)

plots_fc = df_to_fc(df, crs="EPSG:28992")


result = extract_imagecollection_stats(
    df=df,
    image_collection=s2,
    band_names=["NDVI", "EVI"],
    description="ndvi_evi_plot_stats",
    scale=10,
    crs="EPSG:28992",
    folder="GEE_exports",
    file_name_prefix="ndvi_evi_plot_stats",
    include_mean=True,
    include_median=True,
    include_percentiles=(25, 75),
    include_stddev=True,
    include_minmax=False,
    include_count=True,
    keep_geometry=False,
)

EEException: Caller does not have required permission to use project 517222506229. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=517222506229 and then retry. Propagation of the new permission may take a few minutes.

In [ ]:
df